## 1. 爬楼梯

- **难度**：简单  
- **标签**：动态规划

**题目**：假设你正在爬楼梯。需要 n 阶你才能到达楼顶。每次你可以爬 1 或 2 个台阶。你有多少种不同的方法可以爬到楼顶呢？

**示例**：
```
输入：n = 2
输出：2
解释：1 阶 + 1 阶；2 阶

输入：n = 3
输出：3
解释：1+1+1、1+2、2+1
```

**思路**：dp[i] 表示爬到第 i 阶的方法数。最后一步只有两种情况：从 i-1 爬 1 阶、从 i-2 爬 2 阶，故 dp[i] = dp[i-1] + dp[i-2]（斐波那契式）。优化：数组可压缩为 prev1/prev2 两个滚动变量。时间 O(n)，空间 O(1)。

**亮点**：从「最后一步怎么走」出发做状态转移，是线性 DP 的入门范式；状态压缩（滚动变量）在只依赖前两个状态时直接可用。


In [ ]:
class Solution:
    def climbStairs(self, n: int) -> int:
        # n = 1 时，只有一种爬法
        if n == 1:
            return 1

        # prev2 表示前前一个状态（对应 dp[0]）
        prev2 = 1
        # prev1 表示前一个状态（对应 dp[1]）
        prev1 = 1

        # 从第 2 阶开始计算
        for i in range(2, n + 1):
            # 当前状态 = 前一个状态 + 前前一个状态
            current = prev1 + prev2

            # 状态向前滚动：原来的 prev1 变成 prev2
            prev2 = prev1
            prev1 = current

        # 循环结束后，prev1 就是 dp[n]
        return prev1


class Solution2:
    def climbStairs(self, n: int) -> int:
        # dp[i] 表示：爬到第 i 阶楼梯一共有多少种不同的方法
        dp = [0] * (n + 1)
        dp[0] = 1   # 爬到第 0 阶，可理解为"什么都不做"
        dp[1] = 1   # 爬到第 1 阶只有一种方法

        for i in range(2, n + 1):
            # 最后一步：从 i-1 爬 1 阶 或 从 i-2 爬 2 阶
            dp[i] = dp[i - 1] + dp[i - 2]

        return dp[n]


# 测试
sol = Solution()
print(sol.climbStairs(2))    # 2
print(sol.climbStairs(3))    # 3
print(sol.climbStairs(45))   # 1836311903

sol2 = Solution2()
print(sol2.climbStairs(2))   # 2
print(sol2.climbStairs(10))  # 89


## 2. 最大子数组和

- **难度**：中等  
- **标签**：动态规划

**题目**：给你一个整数数组 nums，请你找出一个具有最大和的连续子数组（子数组最少包含一个元素），返回其最大和。子数组是数组中的一个连续部分。

**示例**：
```
输入：nums = [-2, 1, -3, 4, -1, 2, 1, -5, 4]
输出：6
解释：连续子数组 [4, -1, 2, 1] 的和最大，为 6。
```

**思路**：dp[i] 表示以 i 为右端点的最大子数组和。每个位置二选一：接上之前的最优段（dp[i-1] + nums[i]），或从 nums[i] 重新开始（nums[i]），取较大值。答案遍历时取所有 dp[i] 的最大值。时间 O(n)，空间 O(n)（可滚动到 O(1)）。

**亮点**：状态定义「以 i 结尾」是子数组类 DP 的标准姿势——把「枚举所有子数组」O(n²) 降为「每个位置只做一次二选一」O(n)，即 Kadane 算法。


In [ ]:
from typing import List

class Solution:
    def maxSubArray(self, nums: List[int]) -> int:
        # 动态规划
        # 定义 dp[i]：以 i 为右端点的数组的最大子数组和
        n = len(nums)

        dp = [0] * n
        dp[0] = nums[0]

        ans = nums[0]  # 记录所有 dp[i] 中的最大值

        for i in range(1, n):
            # 要么接上 dp[i-1] 的段，要么从 nums[i] 重新开始
            dp[i] = max(dp[i - 1] + nums[i], nums[i])

            ans = max(ans, dp[i])

        return ans


# 测试
sol = Solution()
print(sol.maxSubArray([-2, 1, -3, 4, -1, 2, 1, -5, 4]))  # 6
print(sol.maxSubArray([1]))                              # 1
print(sol.maxSubArray([5, 4, -1, 7, 8]))                 # 23


## 3. 打家劫舍

- **难度**：中等  
- **标签**：动态规划

**题目**：你是一个专业的小偷，计划偷窃沿街的房屋。每间房内都藏有一定的现金，影响你偷窃的唯一制约因素就是相邻的房屋装有相互连通的防盗系统，如果两间相邻的房屋在同一晚上被小偷闯入，系统会自动报警。给定一个代表每个房屋存放金额的非负整数数组，计算你**不触动警报装置**的情况下，一夜之内能够偷窃到的最高金额。

**示例**：
```
输入：nums = [1, 2, 3, 1]
输出：4
解释：偷 1 号（1）和 3 号（3），共 4。

输入：nums = [2, 7, 9, 3, 1]
输出：12
解释：偷 1 号（2）、3 号（9）、5 号（1），共 12。
```

**思路**：dp[i] 表示考虑前 i 间房屋能偷到的最高金额。最后一间房屋：不偷 → dp[i-1]；偷 → 第 i-1 间不能偷，得 dp[i-2] + nums[i]。取较大值。时间 O(n)，空间 O(n)（可滚动到 O(1)）。

**亮点**：经典的「偷 or 不偷」0-1 决策模型——状态转移只依赖前两个状态 dp[i-1]、dp[i-2]，与爬楼梯结构同源，但决策从「求和」变成「取 max」。


In [ ]:
from typing import List

class Solution:
    def rob(self, nums: List[int]) -> int:
        n = len(nums)

        # 如果没有房屋，偷到的金额为 0
        if n == 0:
            return 0

        # 如果只有一间房屋，只能偷这一间
        if n == 1:
            return nums[0]

        # dp[i] 表示考虑前 i 间房屋时，能够偷到的最高金额
        dp = [0] * n
        dp[0] = nums[0]
        dp[1] = max(nums[0], nums[1])

        # 枚举每一间房屋，考虑最后一间房屋偷还是不偷
        for i in range(2, n):
            # 不偷第 i 间：答案是 dp[i - 1]
            # 偷第 i 间：第 i - 1 间不能偷，答案是 dp[i - 2] + nums[i]
            dp[i] = max(dp[i - 1], dp[i - 2] + nums[i])

        return dp[n - 1]


# 测试
sol = Solution()
print(sol.rob([1, 2, 3, 1]))        # 4
print(sol.rob([2, 7, 9, 3, 1]))     # 12
print(sol.rob([0]))                 # 0


## 4. 最长递增子序列

- **难度**：中等  
- **标签**：动态规划

**题目**：给你一个整数数组 nums，找到其中**最长严格递增子序列**的长度。子序列是由数组派生而来的序列，删除（或不删除）数组中的元素而不改变其余元素的顺序。

**示例**：
```
输入：nums = [10, 9, 2, 5, 3, 7, 101, 18]
输出：4
解释：最长递增子序列是 [2, 3, 7, 101]，长度为 4。

输入：nums = [0, 1, 0, 3, 2, 3]
输出：4
输入：nums = [7, 7, 7, 7, 7, 7, 7]
输出：1
```

**思路**：dp[i] 表示以 nums[i] 作为结尾的最长递增子序列长度（初始为 1）。对每个 i，枚举它前面的所有 j：若 nums[j] < nums[i]，则 nums[i] 可以接在 nums[j] 后面，dp[i] = max(dp[i], dp[j] + 1)。答案取所有 dp[i] 的最大值。时间 O(n²)，空间 O(n)。

**亮点**：「以 i 结尾」的状态定义使转移只需考虑「前面哪个元素可以当倒数第二个」；注意 dp[i] 是「必须包含 nums[i]」的局部值，全局答案仍是 max——与最大子数组和的答案维护方式一致。可优化：第二层用二分将复杂度降为 O(n log n)。


In [ ]:
from typing import List

class Solution:
    def lengthOfLIS(self, nums: List[int]) -> int:
        n = len(nums)

        # dp[i] 表示以 nums[i] 作为结尾的最长递增子序列长度
        dp = [1] * n

        # ans 记录所有 dp[i] 中的最大值
        ans = 1

        # 从左到右枚举每个位置作为子序列结尾
        for i in range(n):
            # 枚举 i 前面的所有位置
            for j in range(i):
                # 只有 nums[j] < nums[i]，nums[i] 才能接在 nums[j] 后面
                if nums[j] < nums[i]:
                    dp[i] = max(dp[i], dp[j] + 1)

            # 更新最长递增子序列长度
            ans = max(ans, dp[i])

        return ans


# 测试
sol = Solution()
print(sol.lengthOfLIS([10, 9, 2, 5, 3, 7, 101, 18]))  # 4
print(sol.lengthOfLIS([0, 1, 0, 3, 2, 3]))            # 4
print(sol.lengthOfLIS([7, 7, 7, 7, 7, 7, 7]))         # 1


## 5. 完全平方数

- **难度**：中等  
- **标签**：动态规划

**题目**：给你一个整数 n，返回和为 n 的完全平方数的最少数量。完全平方数是一个整数，其值等于另一个整数的平方（如 1、4、9、16）。

**示例**：
```
输入：n = 12
输出：3
解释：12 = 4 + 4 + 4

输入：n = 13
输出：2
解释：13 = 4 + 9
```

**思路**：dp[i] 表示组成整数 i 所需的最少完全平方数数量（初始化为 n+1 表示无穷大，dp[0] = 0）。对每个 i 从小到大计算，枚举「最后一个选的完全平方数 j*j」：若最后选 j*j，前面还需组成 i-j*j，故 dp[i] = min(dp[i], dp[i-j*j] + 1)。时间 O(n·√n)，空间 O(n)。

**亮点**：「枚举最后一枚」的完全背包思想——把 1, 4, 9, 16… 视为无限供应的物品，问装满 n 的最少件数；min 转移 + 无穷大哨兵（n+1）自动处理不可达状态。


In [ ]:
class Solution:
    def numSquares(self, n: int) -> int:
        # dp[i] 表示组成整数 i 所需要的最少完全平方数数量
        dp = [n + 1] * (n + 1)
        # 组成 0 不需要任何完全平方数
        dp[0] = 0

        # 从小到大计算每个整数的答案
        for i in range(1, n + 1):
            j = 1
            # 枚举最后一个完全平方数 j * j
            while j * j <= i:
                # 如果最后选 j * j，那么前面还需要组成 i - j * j
                dp[i] = min(dp[i], dp[i - j * j] + 1)
                j += 1

        return dp[n]


# 测试
sol = Solution()
print(sol.numSquares(12))   # 3 (4+4+4)
print(sol.numSquares(13))   # 2 (4+9)
print(sol.numSquares(100))  # 1 (100)


## 6. 零钱兑换

- **难度**：中等  
- **标签**：动态规划、完全背包

**题目**：给你一个整数数组 coins 表示不同面额的硬币，以及一个整数 amount 表示总金额。计算并返回可以凑成总金额所需的最少硬币个数。如果没有任何一种硬币组合能组成总金额，返回 -1。每种硬币的数量是无限的。

**示例**：
```
输入：coins = [1, 2, 5], amount = 11
输出：3
解释：11 = 5 + 5 + 1

输入：coins = [2], amount = 3
输出：-1

输入：coins = [1], amount = 0
输出：0
```

**思路**：dp[i] 表示组成金额 i 的最少硬币个数（初始化为 amount+1 表示无穷大，dp[0] = 0）。对每个金额 i，枚举最后使用的一枚硬币 coin（coin ≤ i）：dp[i] = min(dp[i], dp[i-coin] + 1)。最后检查 dp[amount] 是否仍为无穷大，是则返回 -1。时间 O(amount × len(coins))，空间 O(amount)。

**亮点**：完全背包的最少件数模板——「外层金额、内层硬币」与「外层硬币、内层金额」等价；amount+1 作哨兵值同时承担「不可达」判断，避免使用 float('inf') 的比较开销。


In [ ]:
from typing import List

class Solution:
    def coinChange(self, coins: List[int], amount: int) -> int:
        # 动态规划：dp[i]表示组成整数i的最少的硬币个数
        # dp[n] = min(dp[n], dp[n-j]+1), j是coins中小于等于n的硬币

        dp = [amount + 1] * (amount + 1)

        dp[0] = 0

        for i in range(1, amount + 1):
            for coin in coins:
                if coin <= i:
                    dp[i] = min(dp[i], dp[i - coin] + 1)

        return dp[amount] if dp[amount] != amount + 1 else -1


# 测试
sol = Solution()
print(sol.coinChange([1, 2, 5], 11))  # 3
print(sol.coinChange([2], 3))         # -1
print(sol.coinChange([1], 0))         # 0


## 7. 单词拆分

- **难度**：中等  
- **标签**：动态规划、字符串

**题目**：给你一个字符串 s 和一个字符串列表 wordDict 作为字典。如果可以利用字典中出现的一个或多个单词拼接出 s 则返回 true。注意不要求字典中的单词全部使用，且单词可以重复使用。

**示例**：
```
输入：s = "leetcode", wordDict = ["leet", "code"]
输出：true
解释：返回 true 因为 "leetcode" 可以由 "leet" 和 "code" 拼接成。

输入：s = "catsandog", wordDict = ["cats", "dog", "sand", "and", "cat"]
输出：false
```

**思路**：dp[i] 表示 s 的前 i 个字符（s[0:i]）能否由字典单词组成（dp[0] = True 表示空串）。转移时枚举「最后一个单词的起点 j」：若 dp[j] 为 True 且 s[j:i] 在字典中，则 dp[i] = True（找到即可 break）。时间 O(n²)（切片与哈希查询 O(1) 级），空间 O(n)。

**亮点**：「枚举最后一个单词」把拆分问题转为前缀可达性传递——dp[j] 是起点合法性的接力棒；用 set 存字典把查询降到 O(1)，找到立即 break 剪枝。


In [ ]:
class Solution:
    def wordBreak(self, s: str, wordDict: list[str]) -> bool:
        # 使用集合，查询某个单词是否在字典中更快
        words = set(wordDict)

        n = len(s)

        # dp[i] 表示 s 的前 i 个字符能否由字典中的单词组成
        dp = [False] * (n + 1)

        # 空字符串可以认为已经成功拆分
        dp[0] = True

        # 枚举字符串前 i 个字符
        for i in range(1, n + 1):

            # 枚举最后一个单词的起点 j
            for j in range(i):

                # 如果前 j 个字符可以拆分，
                # 并且 s[j:i] 是字典中的单词，
                # 那么前 i 个字符也可以拆分
                if dp[j] and s[j:i] in words:
                    dp[i] = True
                    break

        return dp[n]


# 测试
sol = Solution()
print(sol.wordBreak("leetcode", ["leet", "code"]))                    # True
print(sol.wordBreak("applepenapple", ["apple", "pen"]))               # True
print(sol.wordBreak("catsandog", ["cats", "dog", "sand", "and", "cat"]))  # False


## 8. 最长有效括号

- **难度**：困难  
- **标签**：动态规划、字符串

**题目**：给你一个只包含 '(' 和 ')' 的字符串，找出最长有效（格式正确且连续）括号子串的长度。

**示例**：
```
输入：s = "(()"
输出：2
解释：最长有效括号子串是 "()"

输入：s = ")()())"
输出：4
解释：最长有效括号子串是 "()()"
```

**思路**：dp[i] 表示以 s[i] 结尾的最长有效括号子串长度（dp[i] 只在 s[i] == ')' 时非零）。两种情况：
1. s[i-1] == '('（即 "…()"）：当前一对贡献 2，再接上前面连续的有效段 dp[i-2]
2. s[i-1] == ')'（即 "…))"）：跳过以 i-1 结尾的有效段，向前找 j = i - dp[i-1] - 1；若 s[j] == '(' 则与当前 ')' 配对，dp[i] = dp[i-1] + 2，若 j 前还有有效段再加上 dp[j-1]

答案取所有 dp[i] 的最大值（不一定以最后一个字符结尾）。时间 O(n)，空间 O(n)。

**亮点**：以 ')' 结尾定位 + 两种形态分类讨论——情况 2 的 j = i - dp[i-1] - 1 跳跃式回溯是核心：它跳过整段已匹配区间直达待配对的 '('，一次转移 O(1)。


In [ ]:
class Solution:
    def longestValidParentheses(self, s: str) -> int:
        n = len(s)

        # dp[i]：以 s[i] 结尾的最长有效括号子串长度
        dp = [0] * n

        for i in range(1, n):
            # 只有 ')' 才可能作为有效括号子串的结尾
            if s[i] == ')':

                # 情况 1：...()
                if s[i - 1] == '(':
                    # 当前 () 贡献 2，再接上前面的有效括号
                    dp[i] = 2

                    if i >= 2:
                        dp[i] += dp[i - 2]

                # 情况 2：...))
                else:
                    # 跳过以 i-1 结尾的有效括号串，
                    # 找当前 ')' 可能匹配的 '('
                    j = i - dp[i - 1] - 1

                    if j >= 0 and s[j] == '(':
                        # 当前匹配的一对 + 中间已有的有效括号
                        dp[i] = dp[i - 1] + 2

                        # 如果 j 前面还有连续的有效括号，也接上
                        if j >= 1:
                            dp[i] += dp[j - 1]

        # 最长有效括号不一定以最后一个字符结尾
        return max(dp, default=0)


# 测试
sol = Solution()
print(sol.longestValidParentheses("(()"))      # 2
print(sol.longestValidParentheses(")()())"))   # 4
print(sol.longestValidParentheses(""))         # 0


## 9. 乘积最大子数组

- **难度**：中等  
- **标签**：动态规划

**题目**：给你一个整数数组 nums，找出数组中乘积最大的非空连续子数组（该子数组中至少包含一个数字），并返回该子数组所对应的乘积。一个只包含一个元素的数组的乘积是这个元素的值。

**示例**：
```
输入：nums = [2, 3, -2, 4]
输出：6
解释：子数组 [2, 3] 有最大乘积 6。

输入：nums = [-2, 0, -1]
输出：0
解释：结果不能为 2, 因为 [-2, -1] 不是子数组。
```

**思路**：因为负数会把最大乘积「翻转」成最小，所以同时维护两条状态：max_dp[i] / min_dp[i] = 以 nums[i] 结尾的连续子数组的最大/最小乘积。每个位置三选一：nums[i] 自己、接前一个最大、接前一个最小。全局答案 = max(max_dp[i])。时间 O(n)，空间 O(n)（滚动优化到 O(1)）。

**亮点**：最大/最小双状态互相成就——当前的最小乘积遇到负数就是下一个最大乘积的原料（负负得正）；这是「一维状态不够就加维度」的经典案例。


In [ ]:
from typing import List

class Solution:
    def maxProduct(self, nums: List[int]) -> int:
        n = len(nums)

        # max_dp[i]：以 nums[i] 结尾的连续子数组的最大乘积
        # min_dp[i]：以 nums[i] 结尾的连续子数组的最小乘积
        max_dp = [0] * n
        min_dp = [0] * n

        # 初始化
        max_dp[0] = nums[0]
        min_dp[0] = nums[0]

        # 记录整个数组中的最大乘积
        ans = nums[0]

        for i in range(1, n):
            # 以 nums[i] 结尾，有三种可能：
            # 1. nums[i] 自己作为一个新的子数组
            # 2. 接在前一个最大乘积后面
            # 3. 接在前一个最小乘积后面
            max_dp[i] = max(nums[i], max_dp[i - 1] * nums[i], min_dp[i - 1] * nums[i])
            min_dp[i] = min(nums[i], max_dp[i - 1] * nums[i], min_dp[i - 1] * nums[i])

            # 更新全局最大值
            ans = max(ans, max_dp[i])

        return ans


class Solution2:
    def maxProduct(self, nums: List[int]) -> int:
        # 优化空间 + 考虑正负
        max_product = nums[0]
        min_product = nums[0]

        ans = nums[0]

        for i in range(1, len(nums)):
            prev_max = max_product
            prev_min = min_product

            max_product = max(nums[i], prev_max * nums[i], prev_min * nums[i])
            min_product = min(nums[i], prev_max * nums[i], prev_min * nums[i])

            ans = max(ans, max_product)

        return ans


# 测试
sol = Solution()
print(sol.maxProduct([2, 3, -2, 4]))   # 6
print(sol.maxProduct([-2, 0, -1]))     # 0
print(sol.maxProduct([-2, 3, -4]))     # 24（负负得正）

sol2 = Solution2()
print(sol2.maxProduct([2, 3, -2, 4]))  # 6
print(sol2.maxProduct([-2, 3, -4]))    # 24
